# 02. LlamaParse로 PDF 파싱 실험하기

이 노트북의 목적은 원본 PDF를 바로 최종 CSV로 바꾸는 것이 아닙니다. 먼저 LlamaParse가 PDF의 제목, 표, 페이지, 문단 구조를 어떻게 추출하는지 확인하고, 그 결과를 바탕으로 RAG용 CSV 설계를 결정합니다.

## 지금 단계의 핵심 질문

PDF를 먼저 나눌까요, 아니면 먼저 파싱할까요?

이 프로젝트에서는 **먼저 대표 구간을 샘플 파싱하고, 그 결과를 보고 나중에 나누는 방식**이 맞습니다. 단, 처음부터 전체 PDF를 파싱하지 않고 목차/공통사항/일반 자격 장/FAQ·점수표 구간을 샘플 파싱해서 품질을 확인합니다.

이유는 간단합니다. 원본 PDF를 먼저 임의로 자르면 표가 깨지거나, 비자코드 제목과 설명이 서로 다른 조각으로 떨어지거나, 필요서류 목록의 맥락이 손실될 수 있습니다. 반대로 LlamaParse로 먼저 Markdown/JSON을 만들면 페이지와 레이아웃 정보를 보면서 더 정확하게 섹션을 나눌 수 있습니다.

현재 파일 기준으로 사증민원은 482 PDF 페이지, 체류민원은 383 PDF 페이지입니다. 체류민원은 가로 A4이며 일부 구간은 한 PDF 페이지 안에 문서상 두 쪽이 배치된 형태라서, 나중에 `PDF 페이지 번호`와 `문서에 인쇄된 쪽번호`를 따로 관리해야 합니다.

## 공식 문서 기준으로 이해할 점

LlamaParse의 기본 흐름은 다음과 같습니다.

1. `client.files.create(..., purpose="parse")`로 PDF를 업로드합니다.
2. `client.parsing.parse(...)`로 파싱 작업을 실행합니다.
3. `expand=["markdown"]` 같은 옵션으로 Markdown 결과를 함께 가져옵니다.
4. `page_ranges`를 사용하면 전체가 아니라 일부 페이지만 파싱해서 테스트할 수 있습니다.

주의할 점: LlamaParse의 `Split API`는 RAG 청킹용으로 문단을 나누는 기능이 아닙니다. 여러 문서가 하나의 PDF에 합쳐져 있을 때 문서 단위로 분리하는 기능에 가깝습니다. 우리는 이미 사증민원/체류민원 PDF가 따로 있으므로, 지금 필요한 것은 Split API가 아니라 **샘플 파싱 후 섹션/청크 설계**입니다.

## 이번 노트북에서 하는 일

1. 프로젝트 폴더와 PDF 위치를 찾습니다.
2. API 키가 정상인지 확인합니다.
3. 원본 PDF 목록과 출력 파일명을 확인합니다.
4. 먼저 대표 구간을 샘플 파싱합니다. 앞쪽 목차만 보지 않고 중간 장과 표/FAQ 구간도 함께 봅니다.
5. 샘플 결과의 Markdown/JSON을 확인합니다.
6. 품질이 괜찮으면 전체 PDF 파싱으로 전환합니다.

처음 실행할 때는 `DRY_RUN = True` 상태로 둡니다. 실제 API 호출을 하려면 설정 셀에서 `DRY_RUN = False`로 바꿉니다.

In [29]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    """노트북을 어디서 실행하든 프로젝트 루트를 찾습니다.

    Jupyter를 프로젝트 루트에서 열 수도 있고, notebooks 폴더에서 열 수도 있습니다.
    그래서 현재 작업 폴더부터 부모 폴더를 올라가며 requirements.txt와 data/raw를 찾습니다.
    """
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").exists() and (path / "data" / "raw").exists():
            return path
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"
PARSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:     ", RAW_DIR)
print("PARSED_DIR:  ", PARSED_DIR)

PROJECT_ROOT: /Users/junghwan/Desktop/Vizabridge
RAW_DIR:      /Users/junghwan/Desktop/Vizabridge/data/raw
PARSED_DIR:   /Users/junghwan/Desktop/Vizabridge/data/parsed


In [3]:
# 최초 1회 또는 패키지를 새로 설치해야 할 때만 실행하세요.
# 이미 01번 노트북에서 설치했다면 건너뛰어도 됩니다.
%pip install -r {PROJECT_ROOT / 'requirements.txt'}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 56.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
from dotenv import load_dotenv

# 실제 API 키는 .env에 있고, Git에는 올라가지 않도록 .gitignore에 포함되어 있습니다.
load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("LLAMA_CLOUD_API_KEY")

if not api_key:
    raise RuntimeError(f"LLAMA_CLOUD_API_KEY가 없습니다. {PROJECT_ROOT / '.env'} 파일을 확인하세요.")

# 전체 키를 출력하지 않습니다.
print(f"LLAMA_CLOUD_API_KEY loaded: {api_key[:7]}...{api_key[-4:]}")

LLAMA_CLOUD_API_KEY loaded: llx-HFG...9Tr1


## 원본 PDF 확인

`data/raw/` 안의 PDF를 찾습니다. 지금은 사증민원 매뉴얼과 체류민원 매뉴얼 두 개가 있어야 합니다.

In [31]:
pdf_paths = sorted(RAW_DIR.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(f"PDF 파일이 없습니다: {RAW_DIR}")

for index, pdf_path in enumerate(pdf_paths, start=1):
    size_mb = pdf_path.stat().st_size / 1024 / 1024
    print(f"{index}. {pdf_path.name} ({size_mb:.1f} MB)")

1. 260504 사증민원 자격별 안내 매뉴얼.pdf (11.7 MB)
2. 260504 체류민원 자격별 안내 매뉴얼.pdf (11.7 MB)


In [32]:
def manual_type_from_name(pdf_path: Path) -> str:
    """파일명에서 매뉴얼 유형을 추정합니다."""
    name = pdf_path.stem
    if "사증민원" in name:
        return "visa_issuance"
    if "체류민원" in name:
        return "stay_status"
    return "unknown_manual"

def output_stem(pdf_path: Path, run_mode: str, page_range: str | None = None) -> str:
    """파싱 결과 파일명을 안정적으로 만듭니다.

    같은 PDF라도 샘플 파싱과 전체 파싱 결과는 다른 파일로 저장합니다.
    그래야 실험 결과를 덮어쓰지 않고 비교할 수 있습니다.
    """
    base = f"{manual_type_from_name(pdf_path)}_manual"
    if run_mode == "sample":
        safe_range = (page_range or "sample").replace(",", "_").replace("-", "_").replace(" ", "")
        return f"{base}.sample_pages_{safe_range}"
    return base

SAMPLE_PAGE_RANGES_BY_MANUAL = {
    # 표지/목차/공통사항 + 일반 자격 장 + 중후반 표/서류 구간을 함께 봅니다.
    "visa_issuance": "1-12,150-155,300-306",
    # 체류민원은 가로 A4/2단 배치가 섞여 있어 FAQ와 점수표 구간까지 확인합니다.
    "stay_status": "1-15,150-155,299-306",
}

def sample_page_range_for(pdf_path: Path) -> str:
    """매뉴얼별 대표 샘플 페이지 범위를 반환합니다.

    SAMPLE_PAGE_RANGE_OVERRIDE를 문자열로 지정하면 모든 PDF에 같은 범위를 씁니다.
    빠른 테스트용으로만 쓰고, 기본값 None을 권장합니다.
    """
    override = globals().get("SAMPLE_PAGE_RANGE_OVERRIDE")
    if override:
        return override
    manual_type = manual_type_from_name(pdf_path)
    return SAMPLE_PAGE_RANGES_BY_MANUAL.get(manual_type, "1-10")

for pdf_path in pdf_paths:
    sample_range = sample_page_range_for(pdf_path)
    print(pdf_path.name)
    print("  sample pages:   ", sample_range)
    print("  sample markdown:", PARSED_DIR / f"{output_stem(pdf_path, 'sample', sample_range)}.md")
    print("  full markdown:  ", PARSED_DIR / f"{output_stem(pdf_path, 'full')}.md")

260504 사증민원 자격별 안내 매뉴얼.pdf
  sample markdown: /Users/junghwan/Desktop/Vizabridge/data/parsed/visa_issuance_manual.sample_pages_1_5.md
  full markdown:   /Users/junghwan/Desktop/Vizabridge/data/parsed/visa_issuance_manual.md
260504 체류민원 자격별 안내 매뉴얼.pdf
  sample markdown: /Users/junghwan/Desktop/Vizabridge/data/parsed/stay_status_manual.sample_pages_1_5.md
  full markdown:   /Users/junghwan/Desktop/Vizabridge/data/parsed/stay_status_manual.md


## 파싱 실행 설정

처음에는 아래 설정을 그대로 둡니다.

- `DRY_RUN = True`: API 호출 없이 어떤 작업을 할지만 출력합니다.
- `RUN_MODE = "sample"`: 일부 페이지만 테스트합니다.
- `SAMPLE_PAGE_RANGE_OVERRIDE = None`: 기본값은 PDF별 대표 구간을 자동 사용합니다. 정말 빠르게 1-5페이지만 보고 싶을 때만 `"1-5"`로 바꿉니다.
- `OVERWRITE = False`: 이미 결과가 있으면 건너뜁니다.

샘플 결과가 괜찮으면 다음처럼 바꿉니다.

```python
DRY_RUN = False
RUN_MODE = "full"
```

실제 API 호출은 비용과 시간이 발생할 수 있으므로, 샘플 파싱 결과를 확인한 뒤 전체 파싱으로 넘어갑니다.

In [ ]:
DRY_RUN = False
RUN_MODE = "sample"  # "sample" 또는 "full"
OVERWRITE = False

# 권장값은 None입니다.
# None이면 PDF별 대표 구간을 자동 사용합니다.
# 정말 빠르게 앞 5페이지만 테스트하고 싶을 때만 "1-5"로 바꾸세요.
SAMPLE_PAGE_RANGE_OVERRIDE = None

# 공식 문서의 parse tier 중 고품질 파싱을 우선 사용합니다.
# 비용/속도 테스트가 필요하면 나중에 다른 tier도 비교할 수 있습니다.
PARSE_TIER = "agentic"

# 연구용으로 markdown 외 metadata와 job_metadata도 저장합니다.
# items는 표/블록 단위 분석에 유용하지만 응답이 커질 수 있으므로 필요할 때 켜세요.
EXPAND_FIELDS = ["markdown", "metadata", "job_metadata"]

assert RUN_MODE in {"sample", "full"}
print("DRY_RUN:", DRY_RUN)
print("RUN_MODE:", RUN_MODE)
if RUN_MODE == "sample":
    print("SAMPLE_PAGE_RANGE_OVERRIDE:", SAMPLE_PAGE_RANGE_OVERRIDE)
    print("SAMPLE_PAGE_RANGES_BY_MANUAL:")
    for manual_type, page_range in SAMPLE_PAGE_RANGES_BY_MANUAL.items():
        print(f"  {manual_type}: {page_range}")
else:
    print("SAMPLE_PAGE_RANGES_BY_MANUAL: not used")
print("EXPAND_FIELDS:", EXPAND_FIELDS)

DRY_RUN: False
RUN_MODE: sample
SAMPLE_PAGE_RANGE: 1-5
EXPAND_FIELDS: ['markdown', 'metadata', 'job_metadata']


## LlamaParse 호출 함수

아래 함수는 한 개 PDF를 업로드하고 파싱한 뒤 Markdown과 JSON을 저장합니다.

저장 파일은 두 종류입니다.

- `.md`: 사람이 읽고 품질을 확인하기 좋은 Markdown
- `.parse.json`: 나중에 자동 처리와 디버깅에 쓰는 원본 응답 JSON

Markdown 파일에는 `<!-- page: 1 -->` 같은 페이지 마커를 넣습니다. 나중에 CSV를 만들 때 근거 페이지를 잃지 않기 위해서입니다.

In [20]:
import json
from datetime import datetime, timezone
from llama_cloud import LlamaCloud

def to_plain_dict(value):
    """SDK 응답 객체를 JSON 저장 가능한 dict로 바꿉니다."""
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if hasattr(value, "dict"):
        return value.dict()
    return value

def page_markdown(page) -> str:
    """페이지 객체에서 Markdown 또는 text를 안전하게 꺼냅니다."""
    if isinstance(page, dict):
        return page.get("markdown") or page.get("text") or ""
    return getattr(page, "markdown", None) or getattr(page, "text", "") or ""

def markdown_pages(result) -> list[str]:
    """LlamaParse 응답에서 페이지별 Markdown 목록을 추출합니다."""
    markdown = getattr(result, "markdown", None)
    if markdown is None and isinstance(result, dict):
        markdown = result.get("markdown")
    pages = getattr(markdown, "pages", None)
    if pages is None and isinstance(markdown, dict):
        pages = markdown.get("pages")
    return [page_markdown(page) for page in pages or []]

def build_markdown_document(pdf_path: Path, pages: list[str], run_mode: str, page_range: str | None) -> str:
    """페이지별 Markdown을 하나의 문서로 합치고 메타데이터를 위에 남깁니다."""
    lines = [
        f"# {pdf_path.stem}",
        "",
        f"- source_pdf: {pdf_path.name}",
        f"- manual_type: {manual_type_from_name(pdf_path)}",
        f"- run_mode: {run_mode}",
        f"- page_range: {page_range or 'all'}",
        f"- parsed_at: {datetime.now(timezone.utc).isoformat()}",
        "",
    ]
    for page_number, markdown in enumerate(pages, start=1):
        lines.extend([
            f"\n\n<!-- page: {page_number} -->",
            f"\n## Page {page_number}\n",
            markdown.strip(),
        ])
    return "\n".join(lines).strip() + "\n"

def parse_pdf(client: LlamaCloud, pdf_path: Path) -> None:
    page_range = SAMPLE_PAGE_RANGE if RUN_MODE == "sample" else None
    stem = output_stem(pdf_path, RUN_MODE, page_range)
    markdown_path = PARSED_DIR / f"{stem}.md"
    json_path = PARSED_DIR / f"{stem}.parse.json"

    if not OVERWRITE and markdown_path.exists() and json_path.exists():
        print(f"skip existing: {pdf_path.name} -> {markdown_path.name}")
        return

    print(f"uploading: {pdf_path.name}")
    uploaded_file = client.files.create(file=str(pdf_path), purpose="parse")

    parse_kwargs = {
        "file_id": uploaded_file.id,
        "tier": PARSE_TIER,
        "version": "latest",
        "expand": EXPAND_FIELDS,
        "output_options": {
            "markdown": {
                "tables": {"output_tables_as_markdown": True},
            },
        },
    }

    # 샘플 모드에서는 일부 페이지만 파싱합니다.
    # page range 문법은 PDF의 1-based 페이지 번호를 사용합니다. 예: "1-5", "10", "1,3,5-7"
    if page_range:
        parse_kwargs["input_options"] = {"pdf": {"page_ranges": [{"target_pages": page_range}]}}

    print(f"parsing: {pdf_path.name} / mode={RUN_MODE} / pages={page_range or 'all'}")
    result = client.parsing.parse(**parse_kwargs)

    pages = markdown_pages(result)
    markdown_path.write_text(build_markdown_document(pdf_path, pages, RUN_MODE, page_range), encoding="utf-8")

    payload = {
        "source_pdf": pdf_path.name,
        "manual_type": manual_type_from_name(pdf_path),
        "parsed_at": datetime.now(timezone.utc).isoformat(),
        "run_mode": RUN_MODE,
        "page_range": page_range,
        "parse_tier": PARSE_TIER,
        "expand_fields": EXPAND_FIELDS,
        "result": to_plain_dict(result),
    }
    json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"saved markdown: {markdown_path}")
    print(f"saved json:     {json_path}")
    print(f"pages returned: {len(pages)}")

## 실행 전 작업 미리보기

아래 셀은 현재 설정으로 어떤 파일이 생성될지 보여줍니다. `DRY_RUN=True`일 때는 여기까지만 확인해도 됩니다.

In [21]:
page_range = SAMPLE_PAGE_RANGE if RUN_MODE == "sample" else None
for pdf_path in pdf_paths:
    stem = output_stem(pdf_path, RUN_MODE, page_range)
    print(pdf_path.name)
    print("  mode:    ", RUN_MODE)
    print("  pages:   ", page_range or "all")
    print("  markdown:", PARSED_DIR / f"{stem}.md")
    print("  json:    ", PARSED_DIR / f"{stem}.parse.json")

260504 사증민원 자격별 안내 매뉴얼.pdf
  mode:     sample
  pages:    1-5
  markdown: /Users/junghwan/Desktop/Vizabridge/data/parsed/visa_issuance_manual.sample_pages_1_5.md
  json:     /Users/junghwan/Desktop/Vizabridge/data/parsed/visa_issuance_manual.sample_pages_1_5.parse.json
260504 체류민원 자격별 안내 매뉴얼.pdf
  mode:     sample
  pages:    1-5
  markdown: /Users/junghwan/Desktop/Vizabridge/data/parsed/stay_status_manual.sample_pages_1_5.md
  json:     /Users/junghwan/Desktop/Vizabridge/data/parsed/stay_status_manual.sample_pages_1_5.parse.json


In [11]:
if DRY_RUN:
    print("DRY_RUN=True: API를 호출하지 않았습니다.")
    print("샘플 파싱을 실행하려면 DRY_RUN=False로 바꾼 뒤 이 셀부터 다시 실행하세요.")
else:
    client = LlamaCloud()
    for pdf_path in pdf_paths:
        parse_pdf(client, pdf_path)

DRY_RUN=True: API를 호출하지 않았습니다.
샘플 파싱을 실행하려면 DRY_RUN=False로 바꾼 뒤 이 셀부터 다시 실행하세요.


## 파싱 결과 확인

샘플 파싱이 끝나면 아래 셀로 결과 파일 크기와 첫 부분을 확인합니다.

확인할 포인트:

1. 페이지별 구분이 살아 있는가?
2. 표가 Markdown table로 잘 나오는가?
3. 제목/소제목이 검색 가능한 텍스트로 남아 있는가?
4. 비자코드와 비자명, 필요서류가 같은 섹션 안에서 유지되는가?
5. 고객 질문 의도 컬럼을 만들 수 있을 만큼 맥락이 남아 있는가?

In [22]:
parsed_markdown_files = sorted(PARSED_DIR.glob("*.md"))
if not parsed_markdown_files:
    print("아직 생성된 Markdown 결과가 없습니다.")
else:
    for path in parsed_markdown_files:
        print(path.name, f"({path.stat().st_size / 1024:.1f} KB)")

아직 생성된 Markdown 결과가 없습니다.


In [13]:
if parsed_markdown_files:
    preview_path = parsed_markdown_files[0]
    print("preview:", preview_path.name)
    print("-" * 80)
    print(preview_path.read_text(encoding="utf-8")[:4000])
else:
    print("미리 볼 파일이 없습니다.")

미리 볼 파일이 없습니다.


## 다음 단계 판단

샘플 파싱 결과가 괜찮으면 설정 셀에서 다음처럼 바꿔 전체 파싱을 실행합니다.

```python
DRY_RUN = False
RUN_MODE = "full"
```

전체 파싱 결과가 만들어지면 다음 노트북에서는 `data/parsed/*.md`를 읽어서 섹션 단위로 나눕니다. 그때는 단순 페이지 단위가 아니라, 고객 질문에 답할 수 있는 의미 단위로 나눕니다.

예시:

- `visa_code`: D-8
- `user_intent`: 법인설립, 투자, 외국인 창업
- `question_examples`: 한국에서 법인을 설립하고 싶어요; 외국인이 한국에서 사업하려면 어떤 비자가 필요한가요?
- `requirements`: 원문에서 추출한 요건/서류
- `evidence_quote`: 답변 근거로 보여줄 짧은 원문

이렇게 해야 사용자가 비자코드를 몰라도 RAG 검색이 의도 기반으로 작동합니다.